<a href="https://colab.research.google.com/github/yunusarfat/Bangla-Disaster-Severity-Detection-using-multimodal-/blob/main/ovliviate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate timm albumentations langdetect ftfy regex scikit-learn
print('✅ All installed')

In [ ]:
# Cell 2 — Config  ← EDIT THE 3 PATHS BELOW

import os, re, random, warnings

import numpy as np

import pandas as pd

from PIL import Image, ImageStat

from tqdm.notebook import tqdm

import albumentations as A

from albumentations.pytorch import ToTensorV2

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torch.optim import AdamW

from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings('ignore')



# ══════════════════════════════════════════════════════
# 🔧 EDIT THESE PATHS

DRIVE_ROOT = '/content/drive/MyDrive/datathon'
OUTPUT_DIR = '/content/drive/MyDrive/datathon'

# ══════════════════════════════════════════════════════



TRAIN_CSV = os.path.join(DRIVE_ROOT, 'train.csv')
VAL_CSV   = os.path.join(DRIVE_ROOT, 'validation.csv')
TEST_CSV  = os.path.join(DRIVE_ROOT, 'test.csv')

TRAIN_IMG = os.path.join(DRIVE_ROOT, 'images/train')
VAL_IMG   = os.path.join(DRIVE_ROOT, 'images/validation')
TEST_IMG  = os.path.join(DRIVE_ROOT, 'images/test')



os.makedirs(OUTPUT_DIR, exist_ok=True)



# Hyperparams
SEED         = 42
BATCH        = 16
GRAD_ACCUM   = 2          # effective batch = 32
EPOCHS       = 30
LR           = 1e-5       # increased from original 5e-6
HEAD_LR      = 5e-4
MAX_LEN      = 192        # increased from 128 — captures more text context
IMG_SIZE     = 224
WARMUP_FRAC  = 0.1
PATIENCE     = 6          # increased from 4
LABEL_SMOOTH = 0.2
MIXUP_ALPHA  = 0.4
UNFREEZE_EP  = 15        # unfreeze all backbone layers at this epoch
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'



LABELS     = ['Minimal','Mild','Moderate','Severe','Catastrophic']
label2id   = {l: i for i, l in enumerate(LABELS)}
id2label   = {i: l for i, l in enumerate(LABELS)}



CATEGORIES  = ['Drought','Earthquake','Flood','Human Damage','Landslides',
                'Non Disaster','Tropical Storm','Wildfire']
cat2id      = {c: i for i, c in enumerate(CATEGORIES)}
NUM_CATS    = len(CATEGORIES)
CAT_EMB_DIM = 32



CLIP_MODEL   = 'openai/clip-vit-base-patch32'
BANGLA_MODEL = 'csebuetnlp/banglabert'



def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)



seed_everything(SEED)
print(f'Device: {DEVICE}')
print(f'Config: EPOCHS={EPOCHS}, LR={LR}, HEAD_LR={HEAD_LR}, MAX_LEN={MAX_LEN}, PATIENCE={PATIENCE}')

In [ ]:
# Cell 3 — Text Preprocessing
import unicodedata
_NEWS  = re.compile(r'(breaking[:\s]news|live[:\s]|LIVE[:\s]|exclusive[:\s]|\d{1,2}[:\.\s]\d{2}\s*(am|pm|AM|PM)?|https?://\S+|www\.\S+|[#@]\w+|[\U0001F300-\U0001FFFF]|[@#\*\|\^\~`])', re.IGNORECASE)
_WS    = re.compile(r'\s+')
_PUNCT = re.compile(r'([!?.,;:]){2,}')

def clean_text(text):
    if not isinstance(text, str) or not text.strip(): return 'no description'
    text = unicodedata.normalize('NFKC', text)
    text = _NEWS.sub(' ', text)
    text = _PUNCT.sub(r'\1', text)
    text = _WS.sub(' ', text).strip()
    if len(text) > 1800: text = text[:1800]
    return text if text else 'no description'

def preprocess_df(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    for alias in ['text','caption','description','tweet','content','context']:
        if alias in df.columns and 'context' not in df.columns:
            df.rename(columns={alias:'context'}, inplace=True); break
    if 'context' not in df.columns: df['context'] = ''
    for alias in ['image','img','filename','file','image_name','image_path']:
        if alias in df.columns and 'image_name' not in df.columns:
            df.rename(columns={alias:'image_name'}, inplace=True); break
    if 'image_id' not in df.columns:
        df['image_id'] = df['image_name'].apply(lambda x: os.path.splitext(str(x))[0] if pd.notna(x) else '')
    df['context'] = df['context'].fillna('').apply(clean_text)
    before = len(df)
    df.drop_duplicates(subset=['image_id'], keep='first', inplace=True)
    print(f'  Dedup: {before} -> {len(df)} rows')
    df.reset_index(drop=True, inplace=True)
    return df

print('Text preprocessing ✓')

In [ ]:
# Cell 4 — Image Utilities & Augmentations
CLIP_MEAN = (0.48145466, 0.4578275,  0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

def is_valid_image(img, min_var=50.0):
    try:
        stat = ImageStat.Stat(img.convert('L'))
        return stat.stddev[0] > min_var
    except: return False

def safe_open_image(path):
    try:
        img = Image.open(path).convert('RGB')
        if not is_valid_image(img):
            return Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128,128,128))
        return img
    except: return Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(0,0,0))

AUGMENT_TRAIN = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE), scale=(0.65,1.0), ratio=(0.75,1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1, p=0.6),
    A.GaussNoise(var_limit=(10,50), p=0.3),
    A.MotionBlur(blur_limit=5, p=0.2),
    A.RandomRain(p=0.15),
    A.CoarseDropout(max_holes=6, max_height=32, max_width=32, fill_value=0, p=0.3),
    A.Rotate(limit=15, p=0.3),
    A.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ToTensorV2(),
])
AUGMENT_EVAL = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ToTensorV2(),
])
print('Image utilities ✓')

In [ ]:
# Cell 5 — Load CSVs
print('Loading CSVs ...')
train_df = preprocess_df(pd.read_csv(TRAIN_CSV))
val_df   = preprocess_df(pd.read_csv(VAL_CSV))
test_df  = preprocess_df(pd.read_csv(TEST_CSV))

le = LabelEncoder()
le.fit(train_df['label'].astype(str))
NUM_CLASSES = len(le.classes_)
print('Classes:', list(le.classes_))
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Class distribution:')
print(train_df['label'].value_counts())

In [ ]:
# Cell 6 — Load Processors
print('Loading CLIP processor ...')
clip_processor   = CLIPProcessor.from_pretrained(CLIP_MODEL)
print('Loading BanglaBERT tokenizer ...')
bangla_tokenizer = AutoTokenizer.from_pretrained(BANGLA_MODEL)
print('Processors ready ✓')

In [ ]:
# Cell 7 — Dataset (with Category Embedding support)
def find_image_path(img_dir, name):
    base = os.path.join(img_dir, name)
    if os.path.exists(base): return base
    stem = os.path.splitext(name)[0]
    for ext in ['.jpg','.jpeg','.png','.JPG','.JPEG','.PNG']:
        p = os.path.join(img_dir, stem+ext)
        if os.path.exists(p): return p
    # Search subdirectories (handles fixed dataset folder structure)
    for root_d, dirs, files in os.walk(img_dir):
        for f in files:
            if os.path.splitext(f)[0] == stem:
                return os.path.join(root_d, f)
    return None

class DisasterDataset(Dataset):
    def __init__(self, df, img_dir, augment=False, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = AUGMENT_TRAIN if augment else AUGMENT_EVAL
        self.is_test   = is_test

    def __len__(self): return len(self.df)

    def _load_img(self, row):
        path = find_image_path(self.img_dir, str(row['image_name']))
        img  = safe_open_image(path) if path else Image.new('RGB',(IMG_SIZE,IMG_SIZE),color=(128,128,128))
        return self.transform(image=np.array(img))['image']

    def _tokenise(self, text):
        enc = bangla_tokenizer(text, max_length=MAX_LEN, padding='max_length',
                               truncation=True, return_tensors='pt')
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)

    def __getitem__(self, i):
        row       = self.df.iloc[i]
        pv        = self._load_img(row)
        ids, mask = self._tokenise(str(row['context']))
        cat_idx   = torch.tensor(cat2id.get(str(row.get('category','Flood')), 0), dtype=torch.long)
        if self.is_test:
            return pv, ids, mask, cat_idx, str(row['image_id'])
        lbl = int(le.transform([str(row['label'])])[0])
        return pv, ids, mask, cat_idx, torch.tensor(lbl, dtype=torch.long)

def make_sampler(df):
    label_ids = le.transform(df['label'].astype(str))
    counts    = np.bincount(label_ids)
    weights   = 1.0 / counts[label_ids]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_ds = DisasterDataset(train_df, TRAIN_IMG, augment=True)
val_ds   = DisasterDataset(val_df,   VAL_IMG)
test_ds  = DisasterDataset(test_df,  TEST_IMG, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=make_sampler(train_df),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# Cell 8 — Model: CLIP ViT-L/14 + BanglaBERT + Category + Cross-Modal Attention

class CrossModalAttention(nn.Module):
    def __init__(self, img_dim, txt_dim, attn_dim=512):
        super().__init__()
        # img → attn_dim
        self.img_q = nn.Linear(img_dim, attn_dim, bias=False)
        self.txt_k = nn.Linear(txt_dim, attn_dim, bias=False)
        self.txt_v = nn.Linear(txt_dim, attn_dim, bias=False)
        # txt → attn_dim (reverse)
        self.txt_q = nn.Linear(txt_dim, attn_dim, bias=False)
        self.img_k = nn.Linear(img_dim, attn_dim, bias=False)
        self.img_v = nn.Linear(img_dim, attn_dim, bias=False)

        self.scale   = attn_dim ** -0.5
        self.gate    = nn.Linear(attn_dim, attn_dim)
        self.norm    = nn.LayerNorm(attn_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, img_feat, txt_feat):
        # Image attends to Text
        q  = self.img_q(img_feat).unsqueeze(1)
        k  = self.txt_k(txt_feat).unsqueeze(1)
        v  = self.txt_v(txt_feat).unsqueeze(1)
        a1 = torch.softmax((q @ k.transpose(-2,-1)) * self.scale, dim=-1)
        c1 = (self.dropout(a1) @ v).squeeze(1)

        # Text attends to Image
        q2 = self.txt_q(txt_feat).unsqueeze(1)
        k2 = self.img_k(img_feat).unsqueeze(1)
        v2 = self.img_v(img_feat).unsqueeze(1)
        a2 = torch.softmax((q2 @ k2.transpose(-2,-1)) * self.scale, dim=-1)
        c2 = (a2 @ v2).squeeze(1)

        fused = self.norm(c1 + c2)
        return torch.sigmoid(self.gate(fused)) * fused


class MultimodalCLIPBangla(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Image: CLIP ViT-L/14
        clip_full     = CLIPModel.from_pretrained(CLIP_MODEL)
        self.img_enc  = clip_full.vision_model
        self.img_proj = clip_full.visual_projection
        img_dim       = clip_full.config.projection_dim  # 768
        del clip_full

        # Text: BanglaBERT
        self.txt_enc = AutoModel.from_pretrained(BANGLA_MODEL)
        txt_dim      = self.txt_enc.config.hidden_size   # 768

        # Category Embedding (NEW — gives model disaster type knowledge)
        self.cat_embedding = nn.Embedding(NUM_CATS, CAT_EMB_DIM)

        # Cross-Modal Attention
        ATTN_DIM        = 512
        self.cross_attn = CrossModalAttention(img_dim, txt_dim, ATTN_DIM)

        # Fusion: img(768) + txt(768) + cross(512) + category(32)
        fuse_dim = img_dim + txt_dim + ATTN_DIM + CAT_EMB_DIM
        self.classifier = nn.Sequential(
            nn.Linear(fuse_dim, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.5),
            nn.Linear(512, 256),      nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
        self._freeze_bottom(freeze_img_layers=23, freeze_txt_layers=9)

    def _freeze_bottom(self, freeze_img_layers=23, freeze_txt_layers=9):
        frozen = 0
        for name, p in self.img_enc.named_parameters():
            if any(f'encoder.layers.{i}.' in name for i in range(freeze_img_layers)):
                p.requires_grad = False; frozen += 1
        for name, p in self.txt_enc.named_parameters():
            m = re.search(r'encoder\.layer\.(\d+)\.', name)
            if m and int(m.group(1)) < freeze_txt_layers:
                p.requires_grad = False; frozen += 1
        print(f'  Frozen {frozen} bottom backbone params')

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True
        print('  ✅ All backbone layers unfrozen')

    def forward(self, pixel_values, input_ids, attention_mask, category):
        img_cls  = self.img_enc(pixel_values=pixel_values,interpolate_pos_encoding=True).last_hidden_state[:, 0]
        img_feat = self.img_proj(img_cls)
        txt_feat = self.txt_enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        cross    = self.cross_attn(img_feat, txt_feat)
        cat_feat = self.cat_embedding(category)
        fused    = torch.cat([img_feat, txt_feat, cross, cat_feat], dim=-1)
        return self.classifier(fused)

model     = MultimodalCLIPBangla(NUM_CLASSES).to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total: {total/1e6:.1f}M params | Trainable: {trainable/1e6:.1f}M')

In [ ]:
# Cell 9 — Loss, Optimizer, Scheduler

class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets):
        n  = logits.size(-1)
        lp = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            sm = torch.full_like(lp, self.smoothing/(n-1))
            sm.scatter_(1, targets.unsqueeze(1), 1.0-self.smoothing)
        return -(sm * lp).sum(dim=-1).mean()

def mixup_batch(pv, ids, mask, cat, targets, alpha=MIXUP_ALPHA):
    if alpha <= 0: return pv, ids, mask, cat, targets, targets, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(pv.size(0), device=pv.device)
    return lam*pv + (1-lam)*pv[idx], ids, mask, cat, targets, targets[idx], lam

loss_fn = LabelSmoothingCE(LABEL_SMOOTH)

def make_optimizer(model, backbone_lr=LR, head_lr=HEAD_LR):
    backbone_p = [p for p in list(model.img_enc.parameters()) +
                               list(model.img_proj.parameters()) +
                               list(model.txt_enc.parameters()) if p.requires_grad]
    head_p     = (list(model.cross_attn.parameters()) +
                  list(model.classifier.parameters()) +
                  list(model.cat_embedding.parameters()))
    return AdamW([
        {'params': backbone_p, 'lr': backbone_lr, 'weight_decay': 1e-2},
        {'params': head_p,     'lr': head_lr,     'weight_decay': 1e-3},
    ])

optimizer    = make_optimizer(model)
total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f'Optimizer ready | Total steps: {total_steps} | Warmup: {warmup_steps}')

In [ ]:
# Cell 10 — Train / Eval Functions

def get_metrics(targets, preds):
    return {
        'accuracy' : accuracy_score(targets, preds),
        'f1'       : f1_score(targets, preds, average='weighted', zero_division=0),
        'macro_f1' : f1_score(targets, preds, average='macro',    zero_division=0),
    }

def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss, preds, targets_all = 0.0, [], []
    optimizer.zero_grad()
    for step, (pv, ids, mask, cat, tgt) in enumerate(tqdm(loader, leave=False, desc='Train')):
        pv, ids, mask, cat, tgt = pv.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), cat.to(DEVICE), tgt.to(DEVICE)
        pv_m, ids_m, mask_m, cat_m, tgt_a, tgt_b, lam = mixup_batch(pv, ids, mask, cat, tgt)
        with torch.cuda.amp.autocast():
            logits = model(pv_m, ids_m, mask_m, cat_m)
            loss   = (lam*loss_fn(logits,tgt_a) + (1-lam)*loss_fn(logits,tgt_b)) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item() * GRAD_ACCUM
        preds.extend(logits.argmax(-1).detach().cpu().numpy())
        targets_all.extend(tgt.cpu().numpy())
    return total_loss/len(loader), get_metrics(targets_all, preds)

@torch.no_grad()
def eval_epoch(model, loader, loss_fn=None, is_test=False):
    model.eval()
    total_loss, preds, targets_all, all_probs, all_ids = 0.0, [], [], [], []
    for batch in tqdm(loader, leave=False, desc='Eval'):
        if is_test:
            pv, ids, mask, cat, img_ids = batch
            pv, ids, mask, cat = pv.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), cat.to(DEVICE)
            logits = model(pv, ids, mask, cat)
            all_ids.extend(img_ids)
        else:
            pv, ids, mask, cat, tgt = [x.to(DEVICE) for x in batch]
            logits = model(pv, ids, mask, cat)
            if loss_fn: total_loss += loss_fn(logits, tgt).item()
            targets_all.extend(tgt.cpu().numpy())
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        preds.extend(logits.argmax(-1).cpu().numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    metrics   = get_metrics(targets_all, preds) if targets_all else {}
    return total_loss/max(len(loader),1), metrics, preds, all_probs, all_ids

print('Train/eval functions ✓')

In [ ]:
best_f1  = 0.0

In [ ]:
# Cell 11 — Training Loop with Early Stopping + Unfreeze at Epoch 10
scaler   = torch.cuda.amp.GradScaler()
CKPT     = os.path.join(OUTPUT_DIR, 'best_model.pt')
best_f1  = 0.0
no_imprv = 0
history  = []
unfrozen = False

print(f'Starting training for up to {EPOCHS} epochs ...\n')

for ep in range(1, EPOCHS+1):

    # ── Unfreeze all backbone layers at UNFREEZE_EP ──────────────────────────
    if ep == UNFREEZE_EP and not unfrozen:
        model.unfreeze_all()
        unfrozen  = True
        optimizer = make_optimizer(model, backbone_lr=1e-6, head_lr=3e-4)
        remaining        = EPOCHS - ep + 1
        remaining_steps  = (len(train_loader) // GRAD_ACCUM) * remaining
        scheduler = get_cosine_schedule_with_warmup(optimizer,
                                                    int(remaining_steps*0.05),
                                                    remaining_steps)
        print(f'  >> Epoch {ep}: All layers unfrozen. Backbone LR=1e-6, Head LR=3e-4')

    tr_loss, tr_m               = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
    vl_loss, vl_m, _, _, _      = eval_epoch(model, val_loader, loss_fn)

    gap     = tr_m['macro_f1'] - vl_m['macro_f1']
    overfit = ' ⚠️  OVERFIT' if gap > 0.15 else ''
    mark    = ''

    if vl_m['f1'] > best_f1:
        best_f1  = vl_m['f1']
        no_imprv = 0
        torch.save(model.state_dict(), CKPT)
        mark = ' ← saved ✓'
    else:
        no_imprv += 1

    history.append(dict(ep=ep, tr_loss=tr_loss, tr_acc=tr_m['accuracy'],
                        tr_f1=tr_m['f1'], vl_loss=vl_loss,
                        vl_acc=vl_m['accuracy'], vl_f1=vl_m['f1']))

    print(f'Ep {ep:02d}/{EPOCHS} | '
          f'tr_loss={tr_loss:.3f} tr_acc={tr_m["accuracy"]:.3f} tr_f1={tr_m["f1"]:.3f} | '
          f'val_loss={vl_loss:.3f} val_acc={vl_m["accuracy"]:.3f} val_f1={vl_m["f1"]:.3f} | '
          f'gap={gap:+.3f}{overfit}{mark}')

    if no_imprv >= PATIENCE:
        print(f'Early stopping at epoch {ep}')
        break

print(f'\nBest Val weighted-F1 = {best_f1:.4f}')

In [ ]:
CKPT = "/content/drive/MyDrive/datathon/best_model.pt"

model.load_state_dict(
    torch.load(CKPT, map_location=DEVICE)
)

model.to(DEVICE)
model.eval()

In [ ]:
# Cell 12 — Validation Report on Best Checkpoint
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
_, _, vl_preds, _, _ = eval_epoch(model, val_loader, loss_fn)
model.eval()
all_preds_val, all_tgts_val = [], []
with torch.no_grad():
    for pv, ids, mask, cat, tgt in val_loader:
        logits = model(pv.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), cat.to(DEVICE))
        all_preds_val.extend(logits.argmax(-1).cpu().numpy())
        all_tgts_val.extend(tgt.cpu().numpy())
print('='*60)
print('  BEST CHECKPOINT — Validation Report')
print('='*60)
print(f'  Accuracy     : {accuracy_score(all_tgts_val, all_preds_val):.4f}')
print(f'  F1 (weighted): {f1_score(all_tgts_val, all_preds_val, average="weighted", zero_division=0):.4f}')
print(f'  F1 (macro)   : {f1_score(all_tgts_val, all_preds_val, average="macro",    zero_division=0):.4f}')
print()
print(classification_report(all_tgts_val, all_preds_val, target_names=le.classes_, zero_division=0))

In [ ]:
# Cell 13 — Pseudo-Labeling (run only if val_acc >= 0.68)
# This adds high-confidence test predictions back into training for extra boost

PSEUDO_CONFIDENCE = 0.70 # Reverted to original value
PSEUDO_EPOCHS     = 4    # Reverted to original value
PSEUDO_CKPT       = os.path.join(OUTPUT_DIR, 'best_model_pseudo.pt')

print(f'Running test inference for pseudo-labeling ...')
_, _, _, test_probs_pseudo, _ = eval_epoch(model, test_loader, is_test=True)

max_probs   = test_probs_pseudo.max(axis=1)
pseudo_mask = max_probs >= PSEUDO_CONFIDENCE
n_pseudo    = pseudo_mask.sum()
print(f'High-confidence samples (>={PSEUDO_CONFIDENCE}): {n_pseudo}/{len(test_df)} ({n_pseudo/len(test_df)*100:.1f}%)')

if n_pseudo >= 50:
    pseudo_df          = test_df[pseudo_mask].copy()
    pseudo_df['label'] = le.inverse_transform(test_probs_pseudo[pseudo_mask].argmax(axis=1))
    combined_df        = pd.concat([train_df, pseudo_df], ignore_index=True)
    print(f'Training set: {len(train_df)} -> {len(combined_df)} rows')

    pseudo_train_ds = DisasterDataset(combined_df, TRAIN_IMG, augment=True)
    pseudo_loader   = DataLoader(pseudo_train_ds, batch_size=BATCH,
                                 sampler=make_sampler(combined_df),
                                 num_workers=2, pin_memory=True)

    pseudo_opt = make_optimizer(model, backbone_lr=5e-7, head_lr=1e-4)
    p_steps    = (len(pseudo_loader) // GRAD_ACCUM) * PSEUDO_EPOCHS
    pseudo_sch = get_cosine_schedule_with_warmup(pseudo_opt, 0, p_steps)
    pseudo_scl = torch.cuda.amp.GradScaler()
    best_pseudo = best_f1

    for ep in range(1, PSEUDO_EPOCHS+1):
        tr_loss, tr_m            = train_epoch(model, pseudo_loader, pseudo_opt, pseudo_sch, loss_fn, pseudo_scl)
        vl_loss, vl_m, _, _, _   = eval_epoch(model, val_loader, loss_fn)
        print(f'  Pseudo Ep {ep} | tr_acc={tr_m["accuracy"]:.3f} | val_acc={vl_m["accuracy"]:.3f} val_f1={vl_m["f1"]:.3f}')
        if vl_m['f1'] > best_pseudo:
            best_pseudo = vl_m['f1']
            torch.save(model.state_dict(), PSEUDO_CKPT)
            print(f'  ✅ Pseudo model saved (f1={best_pseudo:.4f})')

    if best_pseudo > best_f1:
        print(f'Pseudo-labeling improved: {best_f1:.4f} -> {best_pseudo:.4f}')
        best_f1 = best_pseudo
        model.load_state_dict(torch.load(PSEUDO_CKPT, map_location=DEVICE))
    else:
        print('Pseudo-labeling did not improve — keeping original best')
        model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
else:
    print(f'Only {n_pseudo} high-confidence samples — skipping pseudo-labeling')
    print('This is normal early in training. Run again after more epochs.')

In [ ]:
# Cell 14 — Multi-Scale TTA Inference (18 passes: 3 scales × 6 transforms)

TTA_TRANSFORMS = [
    # 1. Clean eval (baseline)
    AUGMENT_EVAL,
    # 2. Horizontal flip
    A.Compose([A.HorizontalFlip(p=1.0),
               A.Normalize(mean=CLIP_MEAN, std=CLIP_STD), ToTensorV2()]),
    # 3. Center crop (tight)
    A.Compose([A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.85, 1.0)),
               A.Normalize(mean=CLIP_MEAN, std=CLIP_STD), ToTensorV2()]),
    # 4. Color jitter
    A.Compose([A.ColorJitter(brightness=0.2, contrast=0.2, p=1.0),
               A.Normalize(mean=CLIP_MEAN, std=CLIP_STD), ToTensorV2()]),
    # 5. Crop + flip combo
    A.Compose([A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0)),
               A.HorizontalFlip(p=0.5),
               A.Normalize(mean=CLIP_MEAN, std=CLIP_STD), ToTensorV2()]),
    # 6. Slight rotation (disaster images can be tilted)
    A.Compose([A.Rotate(limit=10, p=1.0),
               A.Normalize(mean=CLIP_MEAN, std=CLIP_STD), ToTensorV2()]),
]

# NOTE: VerticalFlip removed — disaster/satellite images have a natural
# up/down orientation. Vertical flip creates unrealistic samples and
# typically HURTS accuracy on this type of data.

TTA_SCALES  = [224, 256, 288]   # 3 scales
# Scale weights: 224 is native CLIP resolution → highest trust
SCALE_WEIGHTS = [0.5, 0.3, 0.2]

model.eval()
all_probs_tta = None
image_ids_out = []
total_passes  = len(TTA_SCALES) * len(TTA_TRANSFORMS)
pass_num      = 0

for scale_idx, (scale, scale_w) in enumerate(zip(TTA_SCALES, SCALE_WEIGHTS)):
    scale_probs_accum = None

    for tta_idx, tta_tf in enumerate(TTA_TRANSFORMS):
        pass_num += 1
        print(f'TTA pass {pass_num}/{total_passes}  [scale={scale}, transform={tta_idx+1}] ...')

        # Build transform: resize to current scale first, then augment
        if tta_idx == 0:
            # Baseline: just resize to scale + normalize (no augment)
            scaled_tf = A.Compose([
                A.Resize(height=scale, width=scale),
                A.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
                ToTensorV2(),
            ])
        else:
            # Inject Resize(scale) at the front of each augment pipeline
            inner_transforms = tta_tf.transforms  # list of albumentations transforms
            scaled_tf = A.Compose([
                A.Resize(height=scale, width=scale),
                *inner_transforms,
            ])

        tta_ds           = DisasterDataset(test_df, TEST_IMG, is_test=True)
        tta_ds.transform = scaled_tf
        tta_loader       = DataLoader(tta_ds, batch_size=BATCH, shuffle=False,
                                      num_workers=2, pin_memory=True)

        probs_pass, ids_pass = [], []
        with torch.no_grad():
            for pv, ids, mask, cat, img_id in tqdm(tta_loader, desc=f'S{scale}-T{tta_idx+1}', leave=False):
                logits = model(pv.to(DEVICE), ids.to(DEVICE),
                               mask.to(DEVICE), cat.to(DEVICE))
                probs_pass.append(F.softmax(logits, dim=-1).cpu().numpy())
                ids_pass.extend(img_id)

        probs_pass = np.concatenate(probs_pass, axis=0)

        # Accumulate within this scale (uniform weight across transforms)
        if scale_probs_accum is None:
            scale_probs_accum = probs_pass.copy()
            image_ids_out     = ids_pass        # set once, same order every pass
        else:
            scale_probs_accum += probs_pass

    # Average across the 6 transforms for this scale
    scale_probs_accum /= len(TTA_TRANSFORMS)

    # Weighted sum across scales
    if all_probs_tta is None:
        all_probs_tta = scale_w * scale_probs_accum
    else:
        all_probs_tta += scale_w * scale_probs_accum

final_preds = le.inverse_transform(all_probs_tta.argmax(axis=1))

# Save probs for ensemble / temperature scaling
np.save(os.path.join(OUTPUT_DIR, 'test_probs_clip.npy'),  all_probs_tta)
np.save(os.path.join(OUTPUT_DIR, 'test_image_ids.npy'),   np.array(image_ids_out))

print(f'\n✅ Multi-scale TTA done ({total_passes} passes across {len(TTA_SCALES)} scales).')
print('Label distribution:')
print(pd.Series(final_preds).value_counts())

In [ ]:
# Cell 15 — Save CLIP-only submission
sub_clip = pd.DataFrame({'image_id': image_ids_out, 'label': final_preds})
clip_sub_path = os.path.join(OUTPUT_DIR, 'submission_clip_only.csv')
sub_clip.to_csv(clip_sub_path, index=False)
print(f'✅ CLIP-only submission: {clip_sub_path}')